# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [17]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import *
from openai import OpenAI

In [79]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_GPT_OSS = 'gpt-oss'
MODEL = MODEL_GPT

In [5]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

API key looks good so far


In [6]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant answering to the question"},
    {"role": "user", "content": question}
    ]

In [9]:
# Get gpt-4o-mini to answer, with streaming
response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
result = response.choices[0].message.content

In [10]:
display(Markdown(result))

The code you've provided is a Python generator expression using the `yield from` statement in conjunction with a set comprehension. Let's break it down step by step to understand what it does:

### Components of the Code

1. **Set Comprehension**:
   - `{book.get("author") for book in books if book.get("author")}`:
     - This creates a set of unique authors from a collection of `books`.
     - It iterates over each `book` in the `books` iterable.
     - For each `book`, it tries to get the value associated with the "author" key using `book.get("author")`.
     - The `if book.get("author")` condition filters out any books that do not have an "author" key or have a None value for the author.
     - The use of curly braces `{}` indicates the creation of a set, which automatically handles duplicates (if multiple books have the same author).

2. **Yield from**:
   - `yield from` is used to yield all values from an iterable (in this case, a set) one at a time.
   - It allows the generator to yield values produced by another iterable without having to explicitly loop through it.

### Overall Functionality

- The provided code takes a list (or other iterable) of `books`, extracts the authors from each book while ensuring that only valid (non-None) author names are included, and yields each unique author one by one.
- This means if you were to call the generator's `next()` function or iterate through it in a loop, you would receive each unique author's name in turn.

### Example

If you have the following list of books:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},
    {"title": "Book 4", "author": "Author A"},
    {"title": "Book 5", "author": "Author C"},
]
```

The code would yield:

- "Author A"
- "Author B"
- "Author C"

### Why Use This Form

- **Efficiency**: Using a set comprehension ensures that authors are unique without needing additional logic to check for duplicates.
- **Generator**: The use of `yield from` makes it more memory-efficient for larger datasets because it allows processing one author at a time rather than building a large list of authors in memory initially.

In conclusion, this code is a concise and efficient way to extract unique authors from a collection of books, filtering out any books that do not have an author.

In [133]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = MODEL_LLAMA

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
openai = OpenAI()
response = ollama.chat.completions.create(model=MODEL, messages=messages)
result = response.choices[0].message.content

display(Markdown(result))

This line of code is written in Python and utilizes generators, iterators, and dictionary methods.

### Breakdown:

1. `{book.get("author") for book in books if book.get("author")}`: This part is a generator expression within a dictionary comprehension.
   - `for book in books` iterates over the list or object `books`.
   - `if book.get("author")`: Only includes items from `books` that have an "author" key in their dictionaries. If no such key exists, the item is skipped.

2. `.yield from`: This operator in Python 3.3 and later iterates over its input iterable (in this case, the generator expression). Instead of returning one value at a time like a regular function with `return`, it yields all items from each iterable, then pauses execution until called again. This can significantly improve performance for large datasets.

### Example:

To make the code more understandable, let's rewrite it without using `.yield from` first and then use it:
```python
# Without yield from
def get_authors(books):
    authors = []
    for book in books:
        if "author" in book:
            authors.append(book["author"])
    return authors

# Using yield from
def get_authors_yield_from(books):
    def inner():
        for book in books:
            if "author" in book:
                yield from (book["author"] for item in book)
    return inner()
```
The main point here is that the new code will be more memory-efficient and faster, especially for large numbers of items.

For brevity and readability, it can be helpful to understand when and why certain constructs are used. In this case using `.yield from` makes the intent (to yield all authors from each book) much clearer than a verbose solution in `get_authors_yield_from`.

In [134]:
website_url = "https://lubimyczytac.pl"
#website_content = fetch_website_contents(website_url)
#website_links = fetch_website_links(website_url)
MODEL = MODEL_GPT

In [135]:
link_system_prompt = """
You are a helpful assistant useful for find links which are relevant to the user question.
Make a list of links in an order of relevancy to the question starting from the most relevant.
If the link is relative recreate absolute url basing on the base address.
You should respond in JSON as in this example:
{
    "links": [
        {"url": "https://full.url/goes/here/about"},
        {"url": "https://another.full.url/careers"}
    ]
}
"""

question_about_website = "What are top 5 books"

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of all links on the website {url} -
Please decide which of them can have a question relevant information and output them in an order according to the relevancy.
Question: {question_about_website}

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [138]:
def select_relevant_links(url, model_lib):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = model_lib.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(result)
    links = json.loads(result)
    print(links)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [139]:
MODEL = 'gpt-oss'
print(select_relevant_links(website_url, ollama))

Selecting relevant links for https://lubimyczytac.pl by calling gpt-oss



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [115]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url,openai)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [116]:
print(fetch_page_and_all_relevant_links("https://lubimyczytac.pl/top100"))

Selecting relevant links for https://lubimyczytac.pl/top100 by calling gpt-4o-mini
Found 6 relevant links
{'links': [{'type': 'top 100 books', 'url': 'https://lubimyczytac.pl/top100'}, {'type': 'book detail', 'url': 'https://lubimyczytac.pl/ksiazka/5242112/w-ciemnym-glodnym-lesie'}, {'type': 'book detail', 'url': 'https://lubimyczytac.pl/ksiazka/5218066/hazelthorn'}, {'type': 'book detail', 'url': 'https://lubimyczytac.pl/ksiazka/5243182/las-papierowy'}, {'type': 'book detail', 'url': 'https://lubimyczytac.pl/ksiazka/5237779/foliarze-powiesc-teoretycznie-spiskowa'}, {'type': 'book detail', 'url': 'https://lubimyczytac.pl/ksiazka/5239943/prawie-wieczor-wciaz-jasno'}]}
## Landing Page:

Ranking książek - polecane i ciekawe książki Bestsellery Maj 2026 | Lubimyczytać.pl

Szukaj
Anuluj
Katalog
Nowości
Top 100
Cytaty
Aplikacja
Książka roku
Tu możesz zobaczyć wszystkie książki dodane do listy prezentowej.
Zaloguj się
Twórz swoją biblioteczkę, inspiruj się półkami innych osób, dodawaj oceny i

In [123]:
system_prompt = """
You are a helpful assistant that analyzes the contents of several relevant subpages from the main website
to answers the user question.
The response should be base strictly on the provided information.
Respond in markdown without code blocks.
Include the additional details about what paragraph, section or part of the page the information is taken.
"""

In [124]:
def get_user_prompt(url, question_about_website):
    user_prompt = f"""
You are looking for answer the user's question: {question_about_website}
Here are the contents of the page the question regards and
other relevant pages which may contain information necessary for answering.
Use the followed information to answer the question.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [125]:
def answer_question_about_website(url, question):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(url, question)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [127]:
MODEL = MODEL_GPT
answer_question_about_website(website_url, question_about_website)

Selecting relevant links for https://lubimyczytac.pl by calling gpt-4o-mini
{'links': [{'type': 'top 100 ranking', 'url': 'https://lubimyczytac.pl/top100'}, {'type': 'books catalog', 'url': 'https://lubimyczytac.pl/katalog/ksiazki'}, {'type': 'new and upcoming books', 'url': 'https://lubimyczytac.pl/ksiazki/nowosci-i-zapowiedzi'}, {'type': 'book opinion page', 'url': 'https://lubimyczytac.pl/ksiazki/opinie'}, {'type': 'book pages', 'url': 'https://lubimyczytac.pl/ksiazka/5111396/onyx-storm-onyksowa-burza'}]}
Found 5 relevant links


The provided content does not include a direct list or names of the top 5 books in the Top 100 ranking. The pages mention the existence of a "Top 100" ranking and categories for books, but no specific book titles or their ranks are given in the excerpts shared.

To find the top 5 books in the Top 100 ranking, you would need to navigate to the "Top 100" section on the website (likely under the "Top 100" tab/menu) where the ranking of books is published. Unfortunately, this information is not present in the text excerpts provided.

**Summary:**
- The "Top 100" ranking section exists on the website (mentioned in the "Landing Page" and "top 100 ranking" page).
- No explicit list of the top 5 books is given in the text shown.
- You need to visit the "Top 100" part of the catalog or ranking page to see the current top 5 books.

(Information taken from the "Landing Page" and "top 100 ranking" excerpts, no direct listing found.)